In [ ]:
import matplotlib
matplotlib.use("Agg")  # Set backend for plotting in notebooks
from pathlib import Path
import pandas as pd
from risk_validation.core.metrics.impl import pd as pd_metrics
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.plots import plot_gini
from IPython.display import Image, display

# Define data directory
DATA_DIR = Path('..') / 'data'

# Load data
csv_path = DATA_DIR / 'sample.csv'
df = pd.read_csv(csv_path)

# Separate development and validation datasets
# Using 2019 as development year and 2020 as validation year
available_years = sorted(df['score_year'].unique())
dev_year = 2019
val_year = 2020
df_dev = df[df['score_year'] == dev_year].copy()
df_val = df[df['score_year'] == val_year].copy()

# Initialize metric service for Gini
service = PDMetricsService(['gini'])
params = {
    'y_col': 'default_flag',  # Change to your actual target column
    'p_col': 'score_pd',      # Change to your actual probability column
    'score_col': 'score_pd',  # Usually same as p_col for PD models
}

# Compute Gini for development data
result_dev = service.compute(df_dev, params)

# Compute Gini for validation data
result_val = service.compute(df_val, params)

# Get Gini results
print("Gini Coefficient - DEV Year 2019:", result_dev['value'].iloc[0])
print("Gini Coefficient - VAL Year 2020:", result_val['value'].iloc[0])

# Plot Gini
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)
gini_plot_path = plot_gini(
    y_true=df_val['default_flag'],
    y_score=df_val['score_pd'],
    title='Gains Chart for Gini - Year 2020',
    out_path=plot_dir / 'gini_plot.png'
)
display(Image(filename=str(gini_plot_path)))